# LangGraph 011 — Persistence, Threads and Time Travel

The feature the rest of the track is built on. Needs `langgraph`; **no API
key** — the two model calls are scripted and number their drafts, so a replay
is visibly a new run.

| Part | What we check |
|---|---|
| A | the four checkpoints of a two-node graph |
| B | checkpoints follow **super-steps**, not nodes |
| C | two threads, and an unknown one |
| D | crash, then `invoke(None)` — with the calls counted |
| E | time travel: replay, branch, and both `update_state` traps |
| F | a checkpointer that survives a fresh process |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — What is saved

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

# scripted "models" that number their drafts, so a replay is visibly a new run
CALLS = {"joke": 0}

def generate_joke(state: JokeState) -> dict:
    CALLS["joke"] += 1
    return {"joke": f"joke #{CALLS['joke']} about {state['topic']}"}

def generate_explanation(state: JokeState) -> dict:
    return {"explanation": f"explanation of: {state['joke']}"}

graph = StateGraph(JokeState)
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

checkpointer = InMemorySaver()                       # the store
workflow = graph.compile(checkpointer=checkpointer)  # attached at COMPILE time

config = {"configurable": {"thread_id": "1"}}        # the thread, at INVOKE time
workflow.invoke({"topic": "pizza", "joke": "", "explanation": ""}, config=config)

for snapshot in workflow.get_state_history(config):
    print(f"step {snapshot.metadata['step']:>2}  next {str(snapshot.next):<26} "
          f"{sorted(k for k, v in snapshot.values.items() if v)}")

# step  2  next ()                         ['explanation', 'joke', 'topic']
# step  1  next ('generate_explanation',)  ['joke', 'topic']
# step  0  next ('generate_joke',)         ['topic']
# step -1  next ('__start__',)             []
#
# Four checkpoints, newest first. Persistence keeps every intermediate state,
# and each one records which node was about to run - which is what makes
# resuming possible.

In [ ]:
history = list(workflow.get_state_history(config))
assert len(history) == 4                              # two nodes, four checkpoints
assert [s.metadata["step"] for s in history] == [2, 1, 0, -1]
assert history[-1].metadata["source"] == "input"      # the oldest is the input
assert history[0].next == ()                          # the newest has nothing left

# each checkpoint names the node that was about to run
assert history[2].next == ("generate_joke",)
assert history[1].next == ("generate_explanation",)
print("get_state returns the newest:", sorted(
    k for k, v in workflow.get_state(config).values.items() if v))
print("every intermediate state is kept, each with its `next`")

## Part B — Per node, or per super-step?

Three nodes, wired two ways. Guess the two counts before running it.

In [ ]:
# "One checkpoint per super-step" is worth counting rather than believing.
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class Fan(TypedDict):
    log: Annotated[list, operator.add]

def build(parallel, n=3):
    graph = StateGraph(Fan)
    for i in range(n):
        graph.add_node(f"n{i}", (lambda k: lambda s: {"log": [k]})(i))
    if parallel:
        for i in range(n):
            graph.add_edge(START, f"n{i}")
            graph.add_edge(f"n{i}", END)
    else:
        graph.add_edge(START, "n0")
        for i in range(n - 1):
            graph.add_edge(f"n{i}", f"n{i + 1}")
        graph.add_edge(f"n{n - 1}", END)
    return graph.compile(checkpointer=InMemorySaver())

for label, parallel in (("parallel", True), ("sequential", False)):
    app = build(parallel)
    cfg = {"configurable": {"thread_id": label}}
    app.invoke({"log": []}, config=cfg)
    history = list(app.get_state_history(cfg))
    print(f"{label:<11} 3 nodes -> {len(history)} checkpoints, "
          f"highest step {max(s.metadata['step'] for s in history)}")

# parallel    3 nodes -> 3 checkpoints, highest step 1
# sequential  3 nodes -> 5 checkpoints, highest step 3
#
# The same three nodes cost 3 checkpoints side by side and 5 in sequence, so
# checkpoints follow SUPER-STEPS and not nodes. The fan-out of lesson 007 is
# cheaper to persist as well as faster to run.

In [ ]:
counts = {}
for label, parallel in (("parallel", True), ("sequential", False)):
    app = build(parallel)
    cfg = {"configurable": {"thread_id": f"check-{label}"}}
    app.invoke({"log": []}, config=cfg)
    counts[label] = len(list(app.get_state_history(cfg)))
assert counts == {"parallel": 3, "sequential": 5}
print(counts)
print("3 against 5 for the same three nodes: a fan-out is cheaper to persist")
print("as well as faster to run, and on a database every checkpoint is a write.")

## Part C — Threads

In [ ]:
CALLS.update(joke=0, explanation=0)

graph = StateGraph(JokeState)
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)
app = graph.compile(checkpointer=InMemorySaver())

for thread, topic in (("1", "pizza"), ("2", "pasta")):
    cfg = {"configurable": {"thread_id": thread}}
    out = app.invoke({"topic": topic, "joke": "", "explanation": ""}, config=cfg)
    print(f"thread {thread}: {out['joke']}")

one = app.get_state({"configurable": {"thread_id": "1"}}).values
two = app.get_state({"configurable": {"thread_id": "2"}}).values
assert one["topic"] == "pizza" and two["topic"] == "pasta"

unknown = app.get_state({"configurable": {"thread_id": "99"}})
print("an unknown thread ->", unknown.values, "| next:", unknown.next)
assert unknown.values == {}            # empty, and NOT an error
print("so check before trusting a thread's state - a typo looks like a new run")

## Part D — Fault tolerance, with the calls counted

In [ ]:
# A node fails. Does resuming re-run the node that already succeeded?
def crashes_once(state: JokeState) -> dict:
    CALLS["explanation"] = CALLS.get("explanation", 0) + 1
    if CALLS["explanation"] == 1:
        raise RuntimeError("the model timed out")
    return {"explanation": f"explanation of: {state['joke']}"}

graph = StateGraph(JokeState)
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", crashes_once)
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)
workflow = graph.compile(checkpointer=InMemorySaver())

CALLS.update(joke=0, explanation=0)
config = {"configurable": {"thread_id": "crash"}}
try:
    workflow.invoke({"topic": "pizza", "joke": "", "explanation": ""}, config=config)
except RuntimeError as e:
    print("crashed:", e)

snapshot = workflow.get_state(config)
print("saved joke :", snapshot.values["joke"])
print("still to run:", snapshot.next)

print("resumed    :", workflow.invoke(None, config=config)["explanation"])
print("node calls :", CALLS)

# crashed: the model timed out
# saved joke : joke #1 about pizza
# still to run: ('generate_explanation',)
# resumed    : explanation of: joke #1 about pizza
# node calls : {'joke': 1, 'explanation': 2}
#
# `invoke(None, config)` means "carry on from the last checkpoint". The first
# node ran ONCE: its output was already saved, so resuming is not re-running.

In [ ]:
assert CALLS["joke"] == 1               # the finished node ran ONCE
assert CALLS["explanation"] == 2        # the failure, then the retry
assert workflow.get_state(config).next == ()
print("resuming is not re-running: the saved state is the failed node's input")

# and what happens if you invoke with an input instead of None?
CALLS.update(joke=0, explanation=0)
fresh = {"configurable": {"thread_id": "crash-2"}}
try:
    workflow.invoke({"topic": "pizza", "joke": "", "explanation": ""}, config=fresh)
except RuntimeError:
    pass
restarted = workflow.invoke({"topic": "pizza", "joke": "", "explanation": ""},
                            config=fresh)
print("invoke(input) after a crash -> joke calls:", CALLS["joke"])
assert CALLS["joke"] == 2
print("passing an input starts a NEW super-step from the saved state, so the")
print("first node runs again. `None` is what means 'carry on'.")

## Part E — Time travel, and the two traps

In [ ]:
# Replaying from a past checkpoint, and the two traps around update_state.
CALLS.update(joke=0, explanation=0)

joke_graph = StateGraph(JokeState)                       # the working graph again
joke_graph.add_node("generate_joke", generate_joke)
joke_graph.add_node("generate_explanation", generate_explanation)
joke_graph.add_edge(START, "generate_joke")
joke_graph.add_edge("generate_joke", "generate_explanation")
joke_graph.add_edge("generate_explanation", END)
workflow = joke_graph.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "tt"}}
first = workflow.invoke({"topic": "pizza", "joke": "", "explanation": ""},
                        config=config)
print("original:", first["joke"])

history = list(workflow.get_state_history(config))
target = [s for s in history if s.next == ("generate_joke",)][0]   # topic set only

replay = {"configurable": {"thread_id": "tt",
                           "checkpoint_id": target.config["configurable"]["checkpoint_id"]}}
print("replayed:", workflow.invoke(None, config=replay)["joke"])
print("checkpoints:", len(history), "->", len(list(workflow.get_state_history(config))))

# trap 1: update_state needs the snapshot's OWN config, not a hand-built one
try:
    workflow.update_state(replay, {"topic": "samosa"})
except KeyError as e:
    print("hand-built config ->", type(e).__name__, e)

new_config = workflow.update_state(target.config, {"topic": "samosa"})

# trap 2: the update made a BRANCH - you have to run the branch
print("old id ->", workflow.invoke(None, config=target.config)["joke"])
print("new id ->", workflow.invoke(None, config=new_config)["joke"])

# original: joke #1 about pizza
# replayed: joke #2 about pizza
# checkpoints: 4 -> 6
# hand-built config -> KeyError 'checkpoint_ns'
# old id -> joke #3 about pizza
# new id -> joke #4 about samosa
#
# Replaying branches the history rather than overwriting it. update_state
# returns a config pointing at the new branch; invoking the OLD checkpoint id
# regenerates the old topic, which is the mistake that looks like the update
# having been ignored.

In [ ]:
history = list(workflow.get_state_history(config))
assert len(history) > 4                        # the replay branched the thread
topics = {s.values.get("topic") for s in history if s.values.get("topic")}
print("topics living in this one thread:", topics)
assert topics == {"pizza", "samosa"}
print()
print("Both branches are still readable. Time travel adds history; it never")
print("overwrites it - which is exactly what you want from a debugging tool.")

In [ ]:
# update_state writes like a node - so reducers apply to it too.
import operator
from typing import Annotated, TypedDict

class Fan(TypedDict):
    log: Annotated[list, operator.add]

graph = StateGraph(Fan)
graph.add_node("n0", lambda s: {"log": ["n0"]})
graph.add_edge(START, "n0")
graph.add_edge("n0", END)
app = graph.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "upd"}}
app.invoke({"log": ["start"]}, config=config)
print("before:", app.get_state(config).values["log"])
app.update_state(config, {"log": ["manual"]})
print("after :", app.get_state(config).values["log"])

# before: ['start', 'n0']
# after : ['start', 'n0', 'manual']
#
# Appended, not replaced. On a chatbot's message list that means update_state
# ADDS a message - to replace one you need the same id (lesson 010) or a
# RemoveMessage.

In [ ]:
final = app.get_state(config).values["log"]
assert final == ["start", "n0", "manual"]
print("update_state appended, so on a message list it ADDS a message.")
print("To replace one you need the same id (lesson 010) or a RemoveMessage.")

## Part F — Durable persistence

`langgraph-checkpoint-sqlite` is not installed in this environment, so the next
cell defines a small checkpointer over the same `BaseCheckpointSaver`
interface. A Postgres or Redis saver differs in where the bytes go, not in
anything the graph sees.

Note the lock in it. LangGraph writes checkpoints from a thread pool, and one
`sqlite3` connection is not safe for concurrent use — without the lock the
**second** `invoke` fails with *bad parameter or other API misuse*. That is why
real savers manage a connection pool.

In [ ]:
"""A minimal SQLite checkpointer, written to make one claim measurable:
that a database-backed checkpointer keeps a thread's state across processes
while InMemorySaver does not.

langgraph-checkpoint-sqlite is not installed on this machine, so this is a
teaching stand-in built on the same BaseCheckpointSaver interface and the
serializer langgraph already ships. It is deliberately simple: no migrations,
no connection pooling, no namespaces beyond the default.
"""
from __future__ import annotations

import sqlite3
import threading

from langgraph.checkpoint.base import BaseCheckpointSaver, CheckpointTuple

SCHEMA = """
CREATE TABLE IF NOT EXISTS checkpoints (
    thread_id     TEXT NOT NULL,
    checkpoint_ns TEXT NOT NULL DEFAULT '',
    checkpoint_id TEXT NOT NULL,
    parent_id     TEXT,
    type          TEXT,
    checkpoint    BLOB,
    metadata_type TEXT,
    metadata      BLOB,
    rowid_order   INTEGER,
    PRIMARY KEY (thread_id, checkpoint_ns, checkpoint_id)
);
CREATE TABLE IF NOT EXISTS writes (
    thread_id     TEXT NOT NULL,
    checkpoint_ns TEXT NOT NULL DEFAULT '',
    checkpoint_id TEXT NOT NULL,
    task_id       TEXT NOT NULL,
    idx           INTEGER NOT NULL,
    channel       TEXT NOT NULL,
    type          TEXT,
    value         BLOB,
    PRIMARY KEY (thread_id, checkpoint_ns, checkpoint_id, task_id, idx)
);
"""


class SqliteSaver(BaseCheckpointSaver):
    def __init__(self, path: str):
        super().__init__()
        # autocommit: Python 3.12+ raises if commit() runs with no transaction
        self.conn = sqlite3.connect(path, check_same_thread=False,
                                    isolation_level=None)
        # langgraph writes checkpoints from a thread pool, and one sqlite3
        # connection is not safe for concurrent use - without this lock the
        # second invoke fails with "bad parameter or other API misuse"
        self.lock = threading.Lock()
        self.conn.executescript(SCHEMA)
        self._order = 0

    def close(self):
        self.conn.close()

    # ------------------------------------------------------------- writing
    def put(self, config, checkpoint, metadata, new_versions):
        cfg = config["configurable"]
        thread_id = cfg["thread_id"]
        ns = cfg.get("checkpoint_ns", "")
        parent = cfg.get("checkpoint_id")
        ctype, cblob = self.serde.dumps_typed(checkpoint)
        mtype, mblob = self.serde.dumps_typed(dict(metadata))
        with self.lock:
            order = self.conn.execute(
                "SELECT COALESCE(MAX(rowid_order), 0) + 1 FROM checkpoints"
            ).fetchone()[0]
        with self.lock:
            self.conn.execute(
                "INSERT OR REPLACE INTO checkpoints VALUES (?,?,?,?,?,?,?,?,?)",
                (thread_id, ns, checkpoint["id"], parent, ctype, cblob, mtype,
                 mblob, order))
        return {"configurable": {"thread_id": thread_id, "checkpoint_ns": ns,
                                 "checkpoint_id": checkpoint["id"]}}

    def put_writes(self, config, writes, task_id, task_path=""):
        cfg = config["configurable"]
        rows = []
        for idx, (channel, value) in enumerate(writes):
            vtype, vblob = self.serde.dumps_typed(value)
            rows.append((cfg["thread_id"], cfg.get("checkpoint_ns", ""),
                         cfg["checkpoint_id"], task_id, idx, channel, vtype, vblob))
        with self.lock:
            self.conn.executemany(
                "INSERT OR REPLACE INTO writes VALUES (?,?,?,?,?,?,?,?)", rows)

    # ------------------------------------------------------------- reading
    def _writes_for(self, thread_id, ns, checkpoint_id):
        with self.lock:
            rows = self.conn.execute(
                "SELECT task_id, channel, type, value FROM writes "
                "WHERE thread_id=? AND checkpoint_ns=? AND checkpoint_id=? "
                "ORDER BY idx", (thread_id, ns, checkpoint_id)).fetchall()
        return [(task_id, channel, self.serde.loads_typed((vtype, value)))
                for task_id, channel, vtype, value in rows]

    def _tuple(self, row):
        (thread_id, ns, checkpoint_id, parent, ctype, cblob, mtype, mblob, _) = row
        config = {"configurable": {"thread_id": thread_id, "checkpoint_ns": ns,
                                   "checkpoint_id": checkpoint_id}}
        parent_config = None
        if parent:
            parent_config = {"configurable": {"thread_id": thread_id,
                                              "checkpoint_ns": ns,
                                              "checkpoint_id": parent}}
        return CheckpointTuple(
            config=config,
            checkpoint=self.serde.loads_typed((ctype, cblob)),
            metadata=self.serde.loads_typed((mtype, mblob)),
            parent_config=parent_config,
            pending_writes=self._writes_for(thread_id, ns, checkpoint_id))

    def get_tuple(self, config):
        cfg = config["configurable"]
        ns = cfg.get("checkpoint_ns", "")
        with self.lock:
            if cfg.get("checkpoint_id"):
                row = self.conn.execute(
                    "SELECT * FROM checkpoints WHERE thread_id=? AND "
                    "checkpoint_ns=? AND checkpoint_id=?",
                    (cfg["thread_id"], ns, cfg["checkpoint_id"])).fetchone()
            else:
                row = self.conn.execute(
                    "SELECT * FROM checkpoints WHERE thread_id=? AND "
                    "checkpoint_ns=? ORDER BY rowid_order DESC LIMIT 1",
                    (cfg["thread_id"], ns)).fetchone()
        return self._tuple(row) if row else None

    def list(self, config, *, filter=None, before=None, limit=None):
        cfg = (config or {}).get("configurable", {})
        query = "SELECT * FROM checkpoints WHERE thread_id=? AND checkpoint_ns=?"
        args = [cfg.get("thread_id"), cfg.get("checkpoint_ns", "")]
        if before:
            query += " AND rowid_order < (SELECT rowid_order FROM checkpoints " \
                     "WHERE thread_id=? AND checkpoint_ns=? AND checkpoint_id=?)"
            args += [cfg.get("thread_id"), cfg.get("checkpoint_ns", ""),
                     before["configurable"]["checkpoint_id"]]
        query += " ORDER BY rowid_order DESC"
        if limit:
            query += f" LIMIT {int(limit)}"
        with self.lock:
            rows = self.conn.execute(query, args).fetchall()
        for row in rows:
            yield self._tuple(row)

In [ ]:
import os, tempfile

os.chdir(tempfile.mkdtemp())            # keep the database out of the way
cfg = {"configurable": {"thread_id": "durable"}}

written = joke_graph.compile(checkpointer=SqliteSaver("jokes.sqlite3"))
written.invoke({"topic": "pizza", "joke": "", "explanation": ""}, config=cfg)
print("db size:", os.path.getsize("jokes.sqlite3"), "bytes")

# a brand-new saver over the same file, as a new process would create
reread = joke_graph.compile(checkpointer=SqliteSaver("jokes.sqlite3"))
print("sqlite    ->", reread.get_state(cfg).values["joke"],
      "|", len(list(reread.get_state_history(cfg))), "checkpoints")
assert len(list(reread.get_state_history(cfg))) == 4

in_memory = joke_graph.compile(checkpointer=InMemorySaver())
print("in memory ->", in_memory.get_state(cfg).values)
assert in_memory.get_state(cfg).values == {}
print()
print("Same graph, same thread_id, same API. What survives a restart is")
print("decided entirely by where the checkpointer puts its bytes.")

In [ ]:
# and resuming a crash through the durable store
CALLS.update(joke=0, explanation=0)
crash_graph = StateGraph(JokeState)
crash_graph.add_node("generate_joke", generate_joke)
crash_graph.add_node("generate_explanation", crashes_once)
crash_graph.add_edge(START, "generate_joke")
crash_graph.add_edge("generate_joke", "generate_explanation")
crash_graph.add_edge("generate_explanation", END)

cfg = {"configurable": {"thread_id": "durable-crash"}}
first = crash_graph.compile(checkpointer=SqliteSaver("jokes.sqlite3"))
try:
    first.invoke({"topic": "pizza", "joke": "", "explanation": ""}, config=cfg)
except RuntimeError as e:
    print("process A crashed:", e)

second = crash_graph.compile(checkpointer=SqliteSaver("jokes.sqlite3"))
print("process B resumes  ->", second.invoke(None, config=cfg)["explanation"])
assert CALLS["joke"] == 1
print("joke calls across both 'processes':", CALLS["joke"])
print("A crash in one process, a resume in another, one model call paid for.")

## What this notebook established

- A two-node graph writes **4 checkpoints**, each recording the values so far
  and the node about to run.
- Checkpoints follow super-steps: three nodes cost **3** in parallel and **5**
  in sequence.
- An unknown `thread_id` returns an empty state, not an error.
- After a crash, `invoke(None, config)` carried on and the finished node ran
  **once**. Passing an input instead re-ran it.
- A replay **branches**: both `pizza` and `samosa` live in one thread's history.
- `update_state` needs the snapshot's own config (`KeyError: 'checkpoint_ns'`
  otherwise), returns a new `checkpoint_id` that you must invoke, and writes
  **through the reducers**.
- A SQLite checkpointer handed the whole thread to a freshly built saver —
  including resuming a crash — where `InMemorySaver` saw `{}`.

## Exercises

1. Add a third node and predict the checkpoint count before running it.
2. Write `get_state_history` to a table with `metadata['writes']` if your
   version has it. Which node wrote each checkpoint?
3. Branch twice from the same checkpoint. How do you tell the branches apart in
   the history?
4. Give `SqliteSaver` a `delete_thread` and use it to clear one conversation.